# Graph Isomorphism Network (GIN) 

Implementation of GIN from *How Powerful are Graph Neural Networks?* (Xu et al., ICLR 2019).

Two ingredients give GIN maximum discriminative power (as powerful as the Weisfeiler-Lehman test):

**1. Injective neighbor aggregation (Eq. 4.1).** A *sum* aggregator over a multiset is injective (Lemma 5 / Corollary 6), unlike mean (which only keeps the distribution) or max (which collapses the multiset to a set). An MLP then models the universal function $\varphi \circ f$:

$$h_v^{(k)} = \text{MLP}^{(k)}\!\Big( (1 + \epsilon^{(k)})\cdot h_v^{(k-1)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k-1)} \Big)$$

**2. Jumping-Knowledge graph readout (Eq. 4.2).** Earlier layers capture local structure, later layers global structure. To keep all of it, we sum node features *within each layer* and concatenate across layers:

$$h_G = \text{CONCAT}\Big( \text{READOUT}\big(\{h_v^{(k)} \mid v \in G\}\big) \;\big|\; k = 0, 1, \dots, K \Big), \qquad \text{READOUT} = \textstyle\sum$$


In [12]:
%pip install torch_geometric scikit-learn

## 1. The GIN layer (Eq. 4.1)

The paper composes $f^{(k+1)} \circ \varphi^{(k)}$ with a single 2-layer MLP (universal approximation theorem). 

In [ ]:
import torch
import torch.nn as nn


class GINLayer(nn.Module):
    """One GIN message-passing layer implementing Eq. 4.1."""

    def __init__(self, in_dim, out_dim, init_eps=0.0, train_eps=True):
        super().__init__()
        # Two-layer MLP -> universal approximation of phi . f (Corollary 6).
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )
        # epsilon: learnable scalar (Corollary 6 holds for all irrational eps).
        if train_eps:
            self.eps = nn.Parameter(torch.tensor(float(init_eps)))
        else:
            self.register_buffer("eps", torch.tensor(float(init_eps)))

    def forward(self, A, X):
        # A: (N, N) dense adjacency for the whole BATCH (block-diagonal: edges never
        # cross graphs, so A @ X aggregates only within each graph). X: (N, in_dim).
        # A @ X is exactly the SUM aggregator over each node's neighbours. 
        neighbor_sum = A @ X
        out = (1.0 + self.eps) * X + neighbor_sum
        return self.mlp(out)


def get_dense_adjacency(edge_index, num_nodes):
    """Convert a PyG edge_index (2, E) into a dense (N, N) adjacency matrix.

    Called on a batched graph, edge_index is already offset per graph, so the
    result is block-diagonal -- one block per graph in the batch.
    """
    A = torch.zeros((num_nodes, num_nodes))
    A[edge_index[0], edge_index[1]] = 1.0
    return A

## 2. The full GIN model with Jumping-Knowledge readout (Eq. 4.2)

stack $K$ GIN layers, then build the graph embedding by summing node features inside each layer (including the input layer $h^{(0)} = X$) and concatenating across all of them. A final MLP classifies the concatenated representation.

In [ ]:
from torch_geometric.nn import global_add_pool


class GIN(nn.Module):
    """GIN graph classifier with JK-style sum readout (Eq. 4.2)."""

    def __init__(self, in_dim, hidden_dim, num_classes, num_layers=5, dropout=0.5):
        super().__init__()
        # num_layers = 5: the GIN paper's default. Each layer is one hop of message
        # passing, Too few layers underreach those substructures; too many cause over-smoothing (node features
        # converge and become indistinguishable) and overfitting. The JK-concat
        # readout below keeps every layer's output, so the classifier still sees the
        # early-layers features even if the deepest layer over-smooths.
        self.layers = nn.ModuleList()
        self.layers.append(GINLayer(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.layers.append(GINLayer(hidden_dim, hidden_dim))

        # Readout concatenates k = 0 (raw input) ... K layer summaries.
        readout_dim = in_dim + num_layers * hidden_dim

        self.readout_norm = nn.BatchNorm1d(readout_dim)
        self.classifier = nn.Sequential(
            nn.Linear(readout_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, A, X, batch):
        # batch: (N,) vector mapping each node to its graph index within the batch.
        hidden_reps = [X]            # h^(0) = raw node features
        h = X
        for layer in self.layers:
            h = layer(A, h)          # Eq. 4.1
            hidden_reps.append(h)    # h^(k)

        # Eq. 4.2: READOUT = sum over nodes PER GRAPH, then CONCAT across layers.
        # global_add_pool uses `batch` to sum each graph's nodes separately, so a
        # 32-graph batch yields (32, dim) per layer -- not one collapsed vector.
        per_layer_sum = [global_add_pool(h_k, batch) for h_k in hidden_reps]
        graph_embedding = torch.cat(per_layer_sum, dim=-1)   # (num_graphs, readout_dim)
        graph_embedding = self.readout_norm(graph_embedding)
        return self.classifier(graph_embedding)              # (num_graphs, num_classes)

## 3. Load MUTAG

We evaluate with 10-fold cross-validation (Section 4 below), so here we just load the dataset and set the batch size — the train/test folds are created inside the CV loop.

In [15]:
import torch
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

print("Loading MUTAG dataset...")
dataset = TUDataset(root="/tmp/MUTAG", name="MUTAG")

# MUTAG: 188 graphs, 7 one-hot node features (atom types), 2 classes.
print(f"Graphs: {len(dataset)}")
print(f"Node features: {dataset.num_node_features} | classes: {dataset.num_classes}")

# How many graphs per mini-batch. PyG's DataLoader concatenates graphs into one big
# block-diagonal graph and adds a `batch` vector (node -> graph index), which is what
# lets BatchNorm normalize over a population of nodes and global_add_pool pool each
# graph separately. We build the per-fold loaders inside the CV loop below.
BATCH_SIZE = 32


def make_loader(indices, shuffle):
    """DataLoader over a subset of `dataset` given integer indices."""
    subset = dataset[[int(i) for i in indices]]
    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=shuffle)

Loading MUTAG dataset...
Graphs: 188
Node features: 7 | classes: 2


## 4. Train and evaluate with 10-fold cross-validation

 **10-fold stratified cross-validation** split the dataset into 10 class-balanced folds, train a fresh model on 9 and test on the held-out 1, rotating through all 10. We record the test accuracy at every epoch of every fold, then — to avoid optimistically cherry-picking — select the single epoch with the highest accuracy *averaged across folds*, and report that epoch's mean ± standard deviation.

In [ ]:
import numpy as np
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold

torch.manual_seed(0)
np.random.seed(0)

EPOCHS = 100
N_FOLDS = 10


def run_epoch(model, loader, optimizer=None):
    """One pass over `loader`. Trains if an optimizer is given, else evaluates."""
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    for data in loader:
        # Build the batch's block-diagonal adjacency and run it in one forward pass.
        A = get_dense_adjacency(data.edge_index, data.num_nodes)
        logits = model(A, data.x.float(), data.batch)   # (num_graphs, num_classes)
        loss = F.cross_entropy(logits, data.y)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct += int((logits.argmax(dim=-1) == data.y).sum())
        n += data.num_graphs
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader):
    return run_epoch(model, loader, optimizer=None)


# Stratified folds keep each fold's class balance close to the full dataset's.
labels = [int(g.y) for g in dataset]
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)

# fold_curves[f, e] = test accuracy of fold f at epoch e.
fold_curves = np.zeros((N_FOLDS, EPOCHS))

print(f"Running {N_FOLDS}-fold cross-validation ({EPOCHS} epochs each)...\n")
for fold, (train_idx, test_idx) in enumerate(skf.split(labels, labels)):
    train_loader = make_loader(train_idx, shuffle=True)
    test_loader = make_loader(test_idx, shuffle=False)

    # Fresh model + optimizer per fold (no leakage of learned weights across folds).
    model = GIN(in_dim=dataset.num_node_features, hidden_dim=32,
                num_classes=dataset.num_classes, num_layers=5)
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

    for epoch in range(EPOCHS):
        run_epoch(model, train_loader, optimizer)
        scheduler.step()
        _, test_acc = evaluate(model, test_loader)
        fold_curves[fold, epoch] = test_acc

    print(f"Fold {fold + 1:02d} | best test {fold_curves[fold].max() * 100:5.2f}%")


mean_curve = fold_curves.mean(axis=0)
best_epoch = int(mean_curve.argmax())
mean_acc = fold_curves[:, best_epoch].mean()
std_acc = fold_curves[:, best_epoch].std()

print(f"\n{N_FOLDS}-fold CV accuracy: {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}% "
      f"(at epoch {best_epoch + 1})")

Running 10-fold cross-validation (100 epochs each)...

Fold 01 | best test 94.74%
Fold 02 | best test 94.74%
Fold 03 | best test 94.74%
Fold 04 | best test 94.74%
Fold 05 | best test 94.74%
Fold 06 | best test 89.47%
Fold 07 | best test 84.21%
Fold 08 | best test 94.74%
Fold 09 | best test 100.00%
Fold 10 | best test 94.44%

10-fold CV accuracy: 88.33% ± 5.11% (at epoch 85)


## 5. Simple test on the training data

The cross-validation above trains a fresh model per fold and throws it away, so here we train a single final model on **all** the graphs and run a handful of them back through it to eyeball predicted-vs-true labels and the overall accuracy on the data we trained on — a quick "did the model actually learn anything?" sanity check.

(This is measured on the training set, so the accuracy is optimistic — the 10-fold CV above is the honest generalization estimate.)

In [ ]:
import torch
import torch.optim as optim

torch.manual_seed(0)

# Train one final model on ALL the graphs (CV discards its per-fold models).
final_model = GIN(in_dim=dataset.num_node_features, hidden_dim=32,
                  num_classes=dataset.num_classes, num_layers=5)
optimizer = optim.Adam(final_model.parameters(), lr=0.01, weight_decay=5e-4)

full_loader = make_loader(range(len(dataset)), shuffle=True)
print(f"Training final model on all {len(dataset)} graphs", end="")
for epoch in range(EPOCHS):
    run_epoch(final_model, full_loader, optimizer)
    if (epoch + 1) % 20 == 0:
        print(".", end="")
print(" done")

# Accuracy over the whole dataset (no shuffle so indices line up with `dataset`).
eval_loader = make_loader(range(len(dataset)), shuffle=False)
_, train_acc = evaluate(final_model, eval_loader)
print(f"Accuracy on the training set: {train_acc * 100:.2f}%")

# Show predictions for a few individual graphs.
final_model.eval()
print("\nSample predictions (graph => predicted | true):")
with torch.no_grad():
    for i in range(8):
        g = dataset[i]
        A = get_dense_adjacency(g.edge_index, g.num_nodes)
        batch = torch.zeros(g.num_nodes, dtype=torch.long)   # single graph
        logits = final_model(A, g.x.float(), batch)
        pred = int(logits.argmax(dim=-1))
        true_label = int(g.y)
        mark = "OK " if pred == true_label else "BAD"
        print(f"  [{mark}] graph {i:3d} => pred {pred} | true {true_label}")